## (1) Importing modules

In [ ]:


#!/usr/bin/python
# -*- coding: UTF-8 -*-

#__modification time__ = 2026-05-10
#__author__ = Qi Zhou, Helmholtz Centre Potsdam - GFZ German Research Centre for Geosciences
#__find me__ = qi.zhou@gfz.de, qi.zhou.geo@gmail.com, https://github.com/Qi-Zhou-Geo
# Please do not distribute this code without the author's permission

import numpy as np
import pandas as pd

from obspy import UTCDateTime

import torch
import torch.nn as nn
print("PyTorch version:", torch.__version__) # PyTorch version: 1.12.1

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec


# region <add the sys.path to search for custom modules>
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent.parent
sys.path.append(str(project_root))


print(f"Current dir: {current_dir}\n"
      f"Project root: {project_root}")
# endregion


# import the custom functions
from functions.model.lstm_model import LSTM_Attention

## (1) Load the data and see what we will get

### (1-1) Prepare the paramsters

In [ ]:
DF_threshold = 0.5
max_norm_clip = 1
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# note: now we are runing the "H" features -> feature_size=12
model = LSTM_Attention(feature_size=12, device=device, dropout=0.25)
model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)

class_weight_binary = torch.tensor([1 - 0.9, 0.9]).to(device) # 0.9 for event
loss_func = torch.nn.CrossEntropyLoss(reduction="mean", weight=class_weight_binary, label_smoothing=0.05)

In [ ]:
train_dataloader = torch.load(f"{current_dir}/data/2017_dataloader.pt")
test_dataloader = torch.load(f"{current_dir}/data/2020_dataloader.pt")

### (1-2) Model training

In [ ]:
# it takes me 5 minutes to finish this process
for batch_data in train_dataloader.dataLoader():
    # t_features of t to t_{sequence_length}, shape ([batch_size, sequence_length]), float time stamps
    t_features = batch_data['t_features']

    # features of t to t_i, shape ([batch_size, sequence_length, num_features])
    features = batch_data['features']

    # t_target of t_{sequence_length+1}, shape ([batch_size, 1]), float time stamps
    t_target = batch_data['t_target']

    # target of t_{sequence_length+1}, shape ([batch_size, 1]), debris flow probability or label
    target = batch_data['target']



    # pass the seismic features and return the raw logits
    raw_logits = model(features, t_features)  # return the model output logits, shape (batch_size, 2)

    # convert the raw_logits to event probability
    predicted_pro = torch.softmax(raw_logits, dim=1)[:, 1] # for my case, the second column is event probability

    # calculate the current losss
    loss = loss_func(raw_logits, target)
    
    
    # update the gredient
    optimizer.zero_grad()
    loss.backward()
    
    # clip the grad to avoid the explosion
    total_norm = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=max_norm_clip,
        norm_type=2
    )

    optimizer.step()



### (1-3) Model testing

In [ ]:
# it takes me 5 minutes to finish this process
time_stamps = []
actual_class = []
predicted_class = []
predicted_pro_l = []

with torch.no_grad():
    
    for batch_data in test_dataloader.dataLoader():
        # t_features of t to t_{sequence_length}, shape ([batch_size, sequence_length]), float time stamps
        t_features = batch_data['t_features']

        # features of t to t_i, shape ([batch_size, sequence_length, num_features])
        features = batch_data['features']

        # t_target of t_{sequence_length+1}, shape ([batch_size, 1]), float time stamps
        t_target = batch_data['t_target']

        # target of t_{sequence_length+1}, shape ([batch_size, 1]), debris flow probability or label
        target = batch_data['target']
        
        
        
        # pass the seismic features and return the raw logits
        raw_logits = model(features, t_features)  # return the model output logits, shape (batch_size, 2)

        # convert the raw_logits to event probability
        predicted_pro = torch.softmax(raw_logits, dim=1)[:, 1] # for my case, the second column is event probability
        pre_y_label = (predicted_pro >= DF_threshold).float()
        
        
        time_stamps.append(t_target.detach().cpu().numpy().tolist())
        actual_class.append(target.detach().cpu().numpy().tolist())
        predicted_class.append(pre_y_label.detach().cpu().numpy().tolist())
        predicted_pro_l.append(predicted_pro.detach().cpu().numpy().tolist())


## (2) Calculate the confusion matrix

### (2-1) Prepare the data

In [ ]:
time_stamps = np.array(time_stamps).reshape(-1)
time_stamps_str = np.array([UTCDateTime(ts).strftime("%Y-%m-%dT%H:%M:%S") for ts in time_stamps])
actual_class = np.array(actual_class).reshape(-1)
predicted_class = np.array(predicted_class).reshape(-1)
predicted_pro_l = np.array(predicted_pro_l).reshape(-1)

In [ ]:
# return as 
# [
#   [ture negative, false positive],
#   [false negative, ture positive],
# ]
cm_2time2 = confusion_matrix(y_true=np.array(actual_class).reshape(-1), y_pred=np.array(predicted_class).reshape(-1))

disp = ConfusionMatrixDisplay(confusion_matrix=cm_2time2)
disp.plot()

## Question 1: The results seem not good, how can we imrpove it?

### (2-2) Plot the predicted probability

In [ ]:
t1 = "2020-06-29T02:00:00" # manually labeled start
t2 = "2020-06-29T10:00:00" # manually labeled end
t_cd29 = "2020-06-29T05:49:00"

time_mark = {}

for idx in [t1, t2, t_cd29]:
    i = np.where(time_stamps_str == idx)[0][0]
    time_mark[idx] = i


In [ ]:
pro = predicted_pro_l[time_mark[t1]: time_mark[t2]]
x = np.arange(len(pro))
idx_cd29 = time_mark[t_cd29] - time_mark[t1]

plt.plot(x, pro, color="black")
plt.axvline(idx_cd29, color="red", ls="--")
plt.xlabel(f"Time [UTC+0 from {t1}]", fontweight='bold')
plt.ylabel(f"Probability", fontweight='bold')
plt.show()

## Question 2: Can you find the peak amplitude for this event and add the time marker?